In [1]:
# standard libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (precision_recall_curve, 
                             average_precision_score,
                             f1_score, precision_score, 
                             recall_score, confusion_matrix,
                             roc_auc_score)
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.4f' % x)

# load data
fraud_df = pd.read_csv('../data/creditcard.csv')

# recreate all engineered features
fraud_df['Amount_log'] = np.log1p(fraud_df['Amount'])
fraud_df['Hour'] = (fraud_df['Time'] / 3600 % 24).astype(int)
fraud_df = fraud_df.sort_values('Time').reset_index(drop=True)
fraud_df['rolling_avg_amount'] = fraud_df['Amount'].rolling(window=100, min_periods=1).mean()
fraud_df['rolling_std_amount'] = fraud_df['Amount'].rolling(window=100, min_periods=1).std().fillna(0)
fraud_df['rolling_fraud_density'] = fraud_df['Class'].rolling(window=100, min_periods=1).mean() * 100
fraud_df['rolling_txn_count'] = fraud_df['Amount'].rolling(window=100, min_periods=1).count()

print(f"Data loaded: {fraud_df.shape}")

Data loaded: (284807, 37)


In [ ]:
with open('../models/splits.json') as f:
    splits = json.load(f)

train_idx    = splits['train_idx']
validate_idx = splits['validate_idx']

print(f"Train idx:    {train_idx}")
print(f"Validate idx: {validate_idx}")